load_ext autoreload: Evita que tengas que reiniciar el kernel

autoreload 2: Coloca el modo de recarga en modo "Recargar todo"

ANYWIDGET_HMR=1: Establece que Anywidget escuche los cambios que se produzcan en el backend

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


# Importando un Barchart escrito en D3.js

Basado en el notebook [Barchart](https://observablehq.com/@d3/bar-chart/2) de Observable.

El notebook nos muestra el siguiente código D3.js

```js
chart = {
  // Declare the chart dimensions and margins.
  const width = 928;
  const height = 500;
  const marginTop = 30;
  const marginRight = 0;
  const marginBottom = 30;
  const marginLeft = 40;

  // Declare the x (horizontal position) scale.
  const x = d3.scaleBand()
      .domain(d3.groupSort(data, ([d]) => -d.frequency, (d) => d.letter)) // descending frequency
      .range([marginLeft, width - marginRight])
      .padding(0.1);
  
  // Declare the y (vertical position) scale.
  const y = d3.scaleLinear()
      .domain([0, d3.max(data, (d) => d.frequency)])
      .range([height - marginBottom, marginTop]);

  // Create the SVG container.
  const svg = d3.create("svg")
      .attr("width", width)
      .attr("height", height)
      .attr("viewBox", [0, 0, width, height])
      .attr("style", "max-width: 100%; height: auto;");

  // Add a rect for each bar.
  svg.append("g")
      .attr("fill", "steelblue")
    .selectAll()
    .data(data)
    .join("rect")
      .attr("x", (d) => x(d.letter))
      .attr("y", (d) => y(d.frequency))
      .attr("height", (d) => y(0) - y(d.frequency))
      .attr("width", x.bandwidth());

  // Add the x-axis and label.
  svg.append("g")
      .attr("transform", `translate(0,${height - marginBottom})`)
      .call(d3.axisBottom(x).tickSizeOuter(0));

  // Add the y-axis and label, and remove the domain line.
  svg.append("g")
      .attr("transform", `translate(${marginLeft},0)`)
      .call(d3.axisLeft(y).tickFormat((y) => (y * 100).toFixed()))
      .call(g => g.select(".domain").remove())
      .call(g => g.append("text")
          .attr("x", -marginLeft)
          .attr("y", 10)
          .attr("fill", "currentColor")
          .attr("text-anchor", "start")
          .text("↑ Frequency (%)"));

  // Return the SVG element.
  return svg.node();
}
```

Esto genera un gráfico como el siguiente:

![image](..\images\chart.png)

Tambien se usa un dataset titulado *alphabet* de prueba que tiene la frecuencia que se repiten las letras del alfabeto en cierto texto.
Para más detalle sobre como usar los Datasets de VizPro, consultar el notebook [Datasets](dataset.ipynb)

In [1]:
import vizpro.datasets as ds

df_alphabet = ds.get_dataset("alphabet")
df_alphabet.columns


['letter', 'frequency']

In [2]:
from vizpro import CustomWidget
import traitlets

In [3]:
class CustomBarplot(CustomWidget):
    _esm = CustomWidget.createWidgetFromLocalFile(paramList=["data"], filePath=r"..\samples\d3_barchart_original.js")
    
    data = traitlets.List([]).tag(sync=True)

barplot = CustomBarplot(data=df_alphabet.data)

In [4]:
barplot

Esto fue posible gracias a que en el código original modificamos algunas lineas importantes.

```js
function plot(data) {//Principal linea cambiada, necesitamos crear la clase "plot" y especificar los parámetros.

  const marginTop = 30;
  const marginRight = 0;
  const marginBottom = 30;
  const marginLeft = 40;


  d3.select(element).selectAll("*").remove(); //Segunda linea principal, es necesaria para reenderizar cuando haya algún cambio en la data.

  const x = d3.scaleBand()
      .domain(d3.groupSort(data, ([d]) => -d.frequency, (d) => d.letter))
      .range([marginLeft, width - marginRight])
      .padding(0.1);
  

  const y = d3.scaleLinear()
      .domain([0, d3.max(data, (d) => d.frequency)])
      .range([height - marginBottom, marginTop]);


  const svg = d3.select(element) //Última linea principal, es necesario agregar el svg al elemento principal del widget.
      .append("svg") //  Dentro del svg está el grafico y dentro de element todos los atributos de Custom Widget necesarios para mostrarlo en una celda.
      .attr("width", width)
      .attr("height", height)
      .attr("viewBox", [0, 0, width, height])
      .attr("style", "max-width: 100%; height: auto;");


  svg.append("g")
      .attr("fill", "steelblue")
    .selectAll()
    .data(data)
    .join("rect")
      .attr("x", (d) => x(d.letter))
      .attr("y", (d) => y(d.frequency))
      .attr("height", (d) => y(0) - y(d.frequency))
      .attr("width", x.bandwidth());


  svg.append("g")
      .attr("transform", `translate(0,${height - marginBottom})`)
      .call(d3.axisBottom(x).tickSizeOuter(0));


  svg.append("g")
      .attr("transform", `translate(${marginLeft},0)`)
      .call(d3.axisLeft(y).tickFormat((y) => (y * 100).toFixed()))
      .call(g => g.select(".domain").remove())
      .call(g => g.append("text")
          .attr("x", -marginLeft)
          .attr("y", 10)
          .attr("fill", "currentColor")
          .attr("text-anchor", "start")
          .text("↑ Frequency (%)"));

}
```

Si quisieras que tu gráfico sea reutilizable para otros Datasets es necesario especificar más elementos. En el caso del Barchart es necesario indicar el nombre de las columnas que serían nuestro eje X y nuestro Eje Y.

In [5]:
df_explorers = ds.read_dataset(r"https://gist.githubusercontent.com/MatiasMaravi/5e540b21eb04ca980d77199317e56ebb/raw/Explorers.tsv",sep="\t")
class CustomBarplot_2(CustomWidget):
    _esm = CustomWidget.createWidgetFromLocalFile(paramList=["data","x_","y_","pallete"], filePath=r"..\samples\d3_barchart_modified.js")
    
    data = traitlets.List([]).tag(sync=True)
    x_ = traitlets.Unicode("").tag(sync=True)
    y_ = traitlets.Unicode("").tag(sync=True)
    pallete = traitlets.List([]).tag(sync=True)

barplot2 = CustomBarplot_2(data=df_explorers.data, x_="explorer", y_="frequency", pallete=["#1f77b4"])

In [8]:
barplot2

```js
function plot(data,x_,y_,pallete) { //Se definieron los nuevo parámetros
  //La variable "letter" fue cambiada por "x_" y la variable "frequency" fue cambiada por "y_".
  //También se agregó la posibilidad de cambiar el color del barchart pasando una lista de paletas de colores.
  const marginTop = 30;
  const marginRight = 20;
  const marginBottom = 30;
  const marginLeft = 40;

  d3.select(element).selectAll("*").remove();

  const x = d3.scaleBand()
      .domain(d3.groupSort(data, ([d]) => -d[y_], (d) => d[x_]))
      .range([marginLeft, width - marginRight])
      .padding(0.1);
  
  const y = d3.scaleLinear()
      .domain([0, d3.max(data, (d) => d[y_])])
      .range([height - marginBottom, marginTop]);


  const svg = d3.select(element)
      .append("svg")
      .attr("width", width)
      .attr("height", height)
      .attr("viewBox", [0, 0, width, height])
      .attr("style", "max-width: 100%; height: auto;");


  svg.append("g")
    .selectAll()
    .data(data)
    .join("rect")
      .attr("x", (d) => x(d[x_]))
      .attr("y", (d) => y(d[y_]))
      .attr("height", (d) => y(0) - y(d[y_]))
      .attr("width", x.bandwidth())
      .attr("fill", (d, i) => pallete[i % pallete.length]); //Linea agregada para cambiar el color a las barras

  svg.append("g")
      .attr("transform", `translate(0,${height - marginBottom})`)
      .call(d3.axisBottom(x).tickSizeOuter(0));


  svg.append("g")
      .attr("transform", `translate(${marginLeft},0)`)
      .call(d3.axisLeft(y).tickFormat((y) => (y * 100).toFixed()))
      .call(g => g.select(".domain").remove())
      .call(g => g.append("text")
          .attr("x", -marginLeft)
          .attr("y", 10)
          .attr("fill", "currentColor")
          .attr("text-anchor", "start")
          .text("↑ Frequency (%)"));
}
```

Para ver un ejemplo de como usar la paleta de colores que brinda nuestra libreria revisar el notebook [colors](colors.ipynb)